# Flight Delay Prediction — Exploratory Data Analysis (EDA)
**Phase 1: Baseline Models & Temporal Evaluation**

This notebook demonstrates exploratory data profiling, target distribution analysis, and leakage prevention validation on the U.S. DOT flight records.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Add src to python path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from flight_delay.config import Config
from flight_delay.data_loader import DataLoader
from flight_delay.data_validation import DataValidator
from flight_delay.eda import EDARunner

cfg = Config.from_yaml(PROJECT_ROOT / "configs/phase1.yaml", base_dir=PROJECT_ROOT)
print(f"Config loaded. Raw data path: {cfg.raw_path}")

In [ ]:
# Load a sample for rapid interactive inspection
loader = DataLoader(cfg.raw_path, cfg.airlines_path, cfg.airports_path)
df_raw = loader.load_data(sample_size=50000, random_seed=42)
print("Sample shape:", df_raw.shape)
df_raw.head()

In [ ]:
# Validate and clean records
validator = DataValidator(target_name=cfg.target_name, raw_target_col=cfg.raw_target_col)
clean_df, dq_report = validator.clean_and_validate(df_raw)
print("Target Distribution:", dq_report["target_distribution"])

In [ ]:
# Delay rate by scheduled departure hour
clean_df['DEP_HOUR'] = clean_df['SCHEDULED_DEPARTURE'].astype(str).str.zfill(4).str[:2].astype(int)
hour_agg = clean_df.groupby('DEP_HOUR')['ARR_DEL15'].agg(['count', 'mean'])
hour_agg['mean'].plot(kind='bar', figsize=(10, 4), color='#d95f02', title='Arrival Delay Rate by Departure Hour')
plt.ylabel('Delay Rate (>=15 min)')
plt.xlabel('Scheduled Departure Hour')
plt.show()